# B3 - Vanilla GAN on MNIST (given specifications)
- Generator: z(100) -> Dense256 ReLU -> Dense512 ReLU -> Dense1024 ReLU -> Dense784 Tanh -> 28x28
- Batch 128, lr 0.0002, Adam(beta1 = 0.5, beta2 = 0.999), BCE
- **Saturating generator loss**: G minimizes $\log(1 - D(G(z)))$
- **Xavier (Glorot) initialization**, **seed 42**, **5 epochs** (as asked in part b; the spec table recommends 50-100 for good quality)

The discriminator size is not given in the question, so a standard 512 -> 256 LeakyReLU MLP is used.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
tf.keras.utils.set_random_seed(42)      # random seed 42 (python, numpy, tf)

# a. Load and preprocess MNIST -> [-1, 1]
(x_train, _), _ = keras.datasets.mnist.load_data()
x_train = (x_train.astype("float32") - 127.5) / 127.5
x_train = x_train.reshape(-1, 784)

BATCH, LATENT, EPOCHS, LR = 128, 100, 5, 0.0002
dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(60000, seed=42).batch(BATCH, drop_remainder=True)

In [ ]:
# b. Build networks (Xavier init)
xavier = 'glorot_uniform'

class VanillaGAN:
    def __init__(self):
        self.G = keras.Sequential([
            layers.Input((LATENT,)),
            layers.Dense(256, activation='relu', kernel_initializer=xavier),
            layers.Dense(512, activation='relu', kernel_initializer=xavier),
            layers.Dense(1024, activation='relu', kernel_initializer=xavier),
            layers.Dense(784, activation='tanh', kernel_initializer=xavier),
        ], name='generator')
        self.D = keras.Sequential([
            layers.Input((784,)),
            layers.Dense(512, kernel_initializer=xavier), layers.LeakyReLU(0.2),
            layers.Dense(256, kernel_initializer=xavier), layers.LeakyReLU(0.2),
            layers.Dense(1, activation='sigmoid', kernel_initializer=xavier),
        ], name='discriminator')
        self.g_opt = keras.optimizers.Adam(LR, beta_1=0.5, beta_2=0.999)
        self.d_opt = keras.optimizers.Adam(LR, beta_1=0.5, beta_2=0.999)
        self.bce = keras.losses.BinaryCrossentropy()

    @tf.function
    def train_step(self, real):
        z = tf.random.normal((BATCH, LATENT))
        with tf.GradientTape() as d_tape, tf.GradientTape() as g_tape:
            fake = self.G(z, training=True)
            d_real = self.D(real, training=True)
            d_fake = self.D(fake, training=True)
            # Discriminator: BCE(D(real),1) + BCE(D(fake),0)
            d_loss = self.bce(tf.ones_like(d_real), d_real) + self.bce(tf.zeros_like(d_fake), d_fake)
            # Saturating generator loss: minimize log(1 - D(G(z)))
            g_loss = tf.reduce_mean(tf.math.log(1.0 - d_fake + 1e-8))
        self.d_opt.apply_gradients(zip(d_tape.gradient(d_loss, self.D.trainable_variables), self.D.trainable_variables))
        self.g_opt.apply_gradients(zip(g_tape.gradient(g_loss, self.G.trainable_variables), self.G.trainable_variables))
        return d_loss, g_loss

gan = VanillaGAN()
gan.G.summary(); gan.D.summary()

In [ ]:
# Adversarial training loop
d_hist, g_hist = [], []
for epoch in range(EPOCHS):
    d_l, g_l = [], []
    for real in dataset:
        d_loss, g_loss = gan.train_step(real)
        d_l.append(d_loss); g_l.append(g_loss)
    d_hist.append(np.mean(d_l)); g_hist.append(np.mean(g_l))
    print(f"Epoch {epoch+1}/{EPOCHS}: D loss={d_hist[-1]:.4f}  G loss (log(1-D(G(z))))={g_hist[-1]:.4f}")

plt.plot(d_hist, label='D loss'); plt.plot(g_hist, label='G loss'); plt.legend(); plt.xlabel('epoch'); plt.show()

In [ ]:
# c. Generate 25 digits, 5x5 grid
imgs = gan.G(tf.random.normal((25, LATENT)), training=False).numpy().reshape(-1, 28, 28)
imgs = (imgs + 1) / 2
plt.figure(figsize=(5, 5))
for i in range(25):
    plt.subplot(5, 5, i + 1); plt.imshow(imgs[i], cmap='gray'); plt.axis('off')
plt.suptitle('Vanilla GAN - 25 generated digits'); plt.show()

**Note:** with the saturating loss, G learns slowly at the start because $\log(1-D(G(z)))$ has tiny gradients when D easily rejects fakes. After only 5 epochs the digits may still be noisy. Training for 50+ epochs (per the spec table) gives clearer digits.